# 第4章 Notebook：模様の特徴量と比較

対応章: [`../chapters/04_reaction_diffusion_snake.md`](../chapters/04_reaction_diffusion_snake.md)

この notebook は、卒業研究準備セミナーの数値実験用である。上から順に実行すれば、本文で説明した図を再現できる。設定パラメータは上部のセルにまとめてある。乱数は seed を固定している。

## 1. ライブラリ読み込み

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu
from skimage.measure import label

np.random.seed(0)

## 2. 模様画像の生成（外部データなしで動く）

`assets/data/` に画像を置けば差し替えられる（最後の課題セル参照）。

In [ ]:
def make_spots(size=200, n=40, seed=0):
    rng = np.random.default_rng(seed)
    y, x = np.mgrid[0:size, 0:size]
    f = np.zeros((size, size))
    for _ in range(n):
        cy, cx = rng.integers(0, size, 2)
        f += np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * (size * 0.04)**2))
    return f / f.max()

def make_stripes(size=200, period=20, angle=0.0):
    y, x = np.mgrid[0:size, 0:size]
    xr = x * np.cos(angle) + y * np.sin(angle)
    return 0.5 * (1 + np.sin(2 * np.pi * xr / period))

target = make_spots(seed=1)
cand_spots = make_spots(seed=2)
cand_stripes = make_stripes(period=20)

## 3-5. 特徴量：面積比・連結成分数・特徴波長(FFT)

In [ ]:
def features(img):
    g = (img - img.min()) / (img.max() - img.min() + 1e-12)
    thr = threshold_otsu(g)
    binimg = g > thr
    area_ratio = float(binimg.mean())
    n_spot = int(label(binimg).max())
    spec = np.abs(np.fft.fftshift(np.fft.fft2(g - g.mean())))
    cy, cx = np.array(spec.shape) // 2
    yy, xx = np.mgrid[0:spec.shape[0], 0:spec.shape[1]]
    rr = np.hypot(xx - cx, yy - cy).astype(int)
    radial = np.bincount(rr.ravel(), spec.ravel()) / (np.bincount(rr.ravel()) + 1e-12)
    radial[0] = 0
    kpeak = int(np.argmax(radial[1:len(radial)//2]) + 1)
    wavelength = img.shape[0] / kpeak if kpeak > 0 else np.inf
    return dict(area_ratio=round(area_ratio, 3), n_spot=n_spot,
                wavelength=round(float(wavelength), 2))

for name, img in [('target(spots)', target), ('spots2', cand_spots), ('stripes', cand_stripes)]:
    print(f'{name:16s}', features(img))

## 6. なぜ画素差ではなく特徴量か（平行移動テスト）

In [ ]:
shifted = np.roll(target, 3, axis=1)
pixel_L2 = float(np.sum((target - shifted)**2))
ft, fs = features(target), features(shifted)
print('pixel L2 after 3px shift =', round(pixel_L2, 1))
print('feature diff (area, n_spot) =',
      round(abs(ft['area_ratio'] - fs['area_ratio']), 4),
      abs(ft['n_spot'] - fs['n_spot']))

## 7. 特徴量距離による比較

In [ ]:
def feature_distance(fa, fb, w=(1.0, 0.05, 0.1)):
    return (w[0] * abs(fa['area_ratio'] - fb['area_ratio'])
            + w[1] * abs(fa['n_spot'] - fb['n_spot'])
            + w[2] * abs(fa['wavelength'] - fb['wavelength']))

ft = features(target)
for name, img in [('spots2', cand_spots), ('stripes', cand_stripes)]:
    print(f'distance(target, {name}) =', round(feature_distance(ft, features(img)), 3))

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, (name, img) in zip(ax, [('target', target), ('spots2', cand_spots), ('stripes', cand_stripes)]):
    g = (img - img.min()) / (img.max() - img.min() + 1e-12)
    a.imshow(g > threshold_otsu(g), cmap='gray'); a.set_title(name); a.axis('off')
plt.tight_layout(); plt.show()

## 8. 課題（自分で変更する）

1. 重み `w` を変えると、target に近い候補が変わるか調べよ。
2. `assets/data/` に画像を置いて、それを target にして特徴量を測れ（下のセル）。

In [ ]:
# === 課題セル ===
from pathlib import Path
img_path = Path('../assets/data/sample_snake_pattern.png')
if img_path.exists():
    from skimage.io import imread
    from skimage.color import rgb2gray
    im = imread(img_path)
    if im.ndim == 3:
        im = rgb2gray(im[..., :3])
    print('loaded image features:', features(im.astype(float)))
else:
    print('画像が無いので生成模様を使用:', features(make_spots(seed=7)))